# Sistem Case-Based Reasoning (CBR) untuk Analisis Putusan Pengadilan TPPO

**Mata Kuliah:** Penalaran Komputer B
**Domain:** Pidana Khusus - Perdagangan Orang (TPPO)
**Tim:** Moh. Khairul Umam (202310370311448) & Nisrina Nurhafizhah (202310370311321)

Notebook ini mencakup seluruh siklus CBR:
1. **Preprocessing** - Ekstraksi dan pembersihan teks dari PDF
2. **Case Representation** - Ekstraksi metadata dan fitur
3. **Case Retrieval** - TF-IDF dan cosine similarity
4. **Solution Reuse** - Prediksi hasil putusan
5. **Evaluasi** - Metrik retrieval dan prediksi


In [ ]:
﻿# ============================================================

## 01. Preprocessing - Ekstraksi dan Pembersihan Teks PDF


Filter: hanya karakter dengan ukuran font antara 6pt dan 40pt
                (watermark menggunakan Helvetica-Bold >= 45pt)


Kelompokkan berdasarkan posisi y (pembulatan)


Urutkan dari atas ke bawah, lalu kiri ke kanan


Mulai skip saat menemui Disclaimer


Hapus header/watermark umum


Hapus footer halaman


Hapus email/telepon


Hapus baris yang hanya berisi scattered chars dari watermark sisa


Gabungkan dan normalisasi spasi


Bersihkan karakter tunggal yang tersisa di awal/akhir baris


Hapus 1-2 karakter di awal baris yang terisolasi


Ekstrak teks dengan filter font


Pembersihan lanjutan


Validasi


Simpan


02_representation.py


Priority 1: Cari di amar putusan (GADILI)


Priority 2: Jika tidak ditemukan di amar, cari di tuntutan/putusan (0-30 tahun wajar)


Denda: cari pola "denda ... RpX" (targeted)


03_retrieval.py


Bangun TF-IDF dengan semua data


Simpan model


Buat 10 queries: 5 dari data (leave-one-out) + 5 sintetis


Gunakan potongan teks yang lebih deskriptif sebagai query


Untuk leave-one-out, gunakan text_full sebagai query


5 queries sintetis (skenario umum) - lebih deskriptif dengan keywords dari kasus target


Simpan queries


Simpan hasil


Hitung rata-rata


Simpan ke CSV untuk evaluasi


04_reuse.py


Load data


Cari ground truth dari source_case (untuk leave-one-out queries)


Hitung MAE (Mean Absolute Error) untuk pidana


05_evaluation.py


Mean Reciprocal Rank


Filter hanya queries dengan ground truth > 0


Mean Absolute Error


Kategorisasi error


Simpan metrik prediksi


Analisis kegagalan (rejection analysis)


Analisis mengapa gagal


MAE from prediction


In [ ]:
Sesi 1: Preprocessing - Ekstraksi dan Pembersihan Teks Putusan PengadilanMata Kuliah: Penalaran Komputer BDomain: Pidana Khusus - Perdagangan Orangimport osimport reimport pdfplumberimport globBASE_DIR = os.path.dirname(os.path.abspath(__file__))PDF_DIR = os.path.join(BASE_DIR, '..', '..', 'pdf')RAW_DIR = os.path.join(BASE_DIR, '..', 'data', 'raw')os.makedirs(RAW_DIR, exist_ok=True)def extract_clean_text_from_pdf(pdf_path):    try:        with pdfplumber.open(pdf_path) as pdf:            all_lines = []            for page in pdf.pages:                chars = page.chars                if not chars:                    continue                real_chars = [                    c for c in chars                    if 6.0 <= round(c.get('size', 0), 1) <= 40.0                ]                if not real_chars:                    continue                y_lines = {}                for c in real_chars:                    y_key = round(c['y0'], 0)                    if y_key not in y_lines:                        y_lines[y_key] = []                    y_lines[y_key].append(c)                for y_key in sorted(y_lines, reverse=True):                    line_chars = sorted(y_lines[y_key], key=lambda c: c['x0'])                    line_text = ''.join(c['text'] for c in line_chars)                    all_lines.append(line_text)            return '\n'.join(all_lines) if all_lines else ''    except Exception as e:        print(f"  Error ekstraksi PDF: {e}")        return Nonedef post_clean_text(text):    lines = text.split('\n')    cleaned = []    skip_until_empty = False    for line in lines:        stripped = line.strip()        if not stripped:            if skip_until_empty:                skip_until_empty = False            continue        if 'Disclaimer' in stripped:            skip_until_empty = True            continue        if skip_until_empty:            continue        if stripped == 'P U T U S A N':            continue        if stripped == 'M A H K A M A H     A G U N G' or stripped.startswith('M A H K A M A H'):            continue        if 'Direktori Putusan Mahkamah Agung' in stripped:            continue        if 'putusan.mahkamahagung.go.id' in stripped:            continue        if stripped == 'DEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESA':            continue        if re.match(r'^Halaman \d+ dari \d+ halaman', stripped):            continue        if re.match(r'^Halaman \d+$', stripped):            continue        if 'Kepaniteraan Mahkamah Agung' in stripped:            skip_until_empty = True            continue        if re.match(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.(go\.id|com|org)$', stripped):            continue        if re.match(r'^\d{2,3}-\d{3,4}-\d{4}$', stripped):            continue        words = stripped.split()        meaningful = [w for w in words if len(w) > 1 or w.isdigit()]        if len(meaningful) == 0 and len(words) <= 3:            continue        cleaned.append(stripped)    result = '\n'.join(cleaned)    result = re.sub(r' +', ' ', result)    result = re.sub(r'\n{3,}', '\n\n', result)    result = re.sub(r'[ \t]+', ' ', result)    lines2 = result.split('\n')    cleaned2 = []    for line in lines2:        line = re.sub(r'^[a-zA-Z]\s+', '', line)        line = re.sub(r'\s+[a-zA-Z]$', '', line)        line = re.sub(r'^[a-zA-Z]\s[a-zA-Z]\s+', '', line)        if line.strip():            cleaned2.append(line.strip())    return '\n'.join(cleaned2)def validate_content(cleaned_text):    words = cleaned_text.split()    meaningful_words = [w for w in words if len(w) > 2]    ratio = len(meaningful_words) / len(words) if words else 0    return {        'total_words': len(words),        'meaningful_words': len(meaningful_words),        'ratio': ratio,        'is_valid': len(words) >= 200 and ratio >= 0.5    }def main():    pdf_files = sorted(glob.glob(os.path.join(PDF_DIR, '*.pdf')))    print(f"Total PDF ditemukan: {len(pdf_files)}")    results = []    for idx, pdf_path in enumerate(pdf_files, 1):        case_id = f"case_{idx:03d}"        pdf_name = os.path.basename(pdf_path)        print(f"[{idx:2d}/70] {pdf_name}", end=' ')        raw_text = extract_clean_text_from_pdf(pdf_path)        if not raw_text or len(raw_text.strip()) == 0:            print("GAGAL - teks kosong")            continue        clean = post_clean_text(raw_text)        validation = validate_content(clean)        status = "OK" if validation['is_valid'] else "LOW"        print(f"-> {validation['total_words']} kata, rasio={validation['ratio']:.1%} [{status}]")        output_path = os.path.join(RAW_DIR, f"{case_id}.txt")        with open(output_path, 'w', encoding='utf-8') as f:            f.write(clean)        results.append({            'case_id': case_id,            'file': pdf_name,            'total_words': validation['total_words'],            'ratio': validation['ratio'],            'status': status        })        if idx % 10 == 0:            print(f"  --- {idx} selesai ---")    print("\n" + "=" * 50)    print("HASIL PREPROCESSING")    print("=" * 50)    ok = [r for r in results if r['status'] == 'OK']    low = [r for r in results if r['status'] == 'LOW']    print(f"Berhasil: {len(ok)}/{len(results)}")    print(f"Peringatan (< 200 kata): {len(low)}")    for r in low:        print(f"  - {r['file']}: {r['total_words']} kata")    print(f"\nFile bersih tersimpan di: {RAW_DIR}")if __name__ == '__main__':    main()Sesi 2: Case Representation - Ekstraksi Metadata dan FiturMata Kuliah: Penalaran Komputer Bimport osimport reimport globimport pandas as pdBASE_DIR = os.path.dirname(os.path.abspath(__file__))RAW_DIR = os.path.join(BASE_DIR, '..', 'data', 'raw')PROCESSED_DIR = os.path.join(BASE_DIR, '..', 'data', 'processed')os.makedirs(PROCESSED_DIR, exist_ok=True)def parse_nomor_perkara(text):    m = re.search(r'Nomor\s+([\d]+/[A-Za-z/.]+/\d{4})', text)    return m.group(1) if m else ''def parse_nama_terdakwa(text):    lines = text.split('\n')    nama_lines = []    capture = False    for line in lines:        stripped = line.strip()        if stripped.startswith('Nama :') or stripped.startswith('Nama:'):            capture = True            nama_lines.append(stripped)        elif capture and stripped.startswith('Tempat Lahir'):            break        elif capture and stripped:            nama_lines.append(stripped)        elif capture and not stripped:            break    return ' '.join(nama_lines).replace('Nama :', '').replace('Nama:', '').strip()def parse_pasal(text):    pasal_pattern = re.findall(        r'Pasal\s+(\d+(?:\s+Ayat\s+\(\d+\))?(?:\s+(?:juncto|dan|atau)\s+Pasal\s+\d+(?:\s+Ayat\s+\(\d+\))?)*)',        text, re.IGNORECASE    )    unique_pasal = []    for p in pasal_pattern:        normalized = re.sub(r'\s+', ' ', p.strip())        if normalized not in unique_pasal:            unique_pasal.append(normalized)    return '; '.join(unique_pasal[:10]) if unique_pasal else ''def parse_pihak(text):    jaksa = ''    terdakwa = ''    m_jaksa = re.search(r'Penuntut Umum pada (Kejaksaan[^;]+)', text)    if m_jaksa:        jaksa = m_jaksa.group(1).strip()    m_terdakwa = re.search(r'Nama\s*:\s*([^;]+)', text)    if m_terdakwa:        terdakwa = m_terdakwa.group(1).strip()    return f"Jaksa: {jaksa} | Terdakwa: {terdakwa}"def parse_ringkasan_fakta(text):    start = text.find('didakwa dengan dakwaan sebagai berikut:')    if start == -1:        start = text.find('dakwaan sebagai berikut:')    if start == -1:        start = text.find('Dakwaan:')    end = text.find('Membaca Tuntutan Pidana')    if end == -1:        end = text.find('Membaca Putusan')    if end == -1:        end = text.find('Mahkamah Agung tersebut;')    if start != -1 and end != -1 and end > start:        return text[start:end].strip()    return ''def parse_ringkasan_putusan(text):    markers = ['G A D I L I:', 'G A D I L I;', 'M E N G A D I L I:', 'MENGADILI:', 'G A D I L I']    found = -1    for marker in markers:        pos = text.rfind(marker)        if pos > found:            found = pos    if found == -1:        return ''    end_markers = ['Demikianlah diputuskan', 'Demikian diputuskan']    end_pos = len(text)    for m in end_markers:        pos = text.find(m, found)        if pos != -1 and pos < end_pos:            end_pos = pos    return text[found:end_pos].strip()def parse_pidana_dari_teks(text, amar_text):    pidana_tahun = 0    denda_rp = 0    if amar_text:        m = re.search(r'pidana penjara (?:selama )?(\d+)', amar_text, re.IGNORECASE)        if m:            pidana_tahun = int(m.group(1))    if pidana_tahun == 0:        pidana_candidates = re.findall(r'pidana penjara (?:selama )?(\d+)', text, re.IGNORECASE)        for p in pidana_candidates:            val = int(p)            if 1 <= val <= 80:                pidana_tahun = val                break    denda_pattern = re.finditer(        r'denda\s+(?:terhadap|sebesar|masing-masing)?\s*(?:sebesar\s+)?\s*Rp([\d.,]+)',        text, re.IGNORECASE    )    for m in denda_pattern:        d_str = m.group(1)        parts = d_str.split(',')        integer_part = parts[0].replace('.', '')        try:            val = int(integer_part)            if 50000000 <= val <= 50000000000:                denda_rp = max(denda_rp, val)        except ValueError:            pass    return pidana_tahun, denda_rpdef parse_tanggal(text):    m = re.search(r'(?:hari\s+\w+,\s+)?tanggal\s+(\d{1,2}\s+\w+\s+\d{4})', text)    if m:        return m.group(1)    m2 = re.search(r'(\d{1,2}\s+\w+\s+\d{4})\s*telah\s*', text)    if m2:        return m2.group(1)    return ''def main():    raw_files = sorted(glob.glob(os.path.join(RAW_DIR, 'case_*.txt')))    print(f"Total file teks ditemukan: {len(raw_files)}")    cases = []    for idx, filepath in enumerate(raw_files, 1):        case_id = f"case_{idx:03d}"        print(f"[{idx:2d}/70] {case_id}")        with open(filepath, 'r', encoding='utf-8') as f:            text = f.read()        no_perkara = parse_nomor_perkara(text)        nama_terdakwa = parse_nama_terdakwa(text)        tanggal = parse_tanggal(text)        pasal = parse_pasal(text)        pihak = parse_pihak(text)        ringkasan_fakta = parse_ringkasan_fakta(text)        ringkasan_putusan = parse_ringkasan_putusan(text)        pidana_tahun, denda_rp = parse_pidana_dari_teks(text, ringkasan_putusan)        word_count = len(text.split())        cases.append({            'case_id': case_id,            'no_perkara': no_perkara,            'tanggal': tanggal,            'nama_terdakwa': nama_terdakwa,            'ringkasan_fakta': ringkasan_fakta,            'ringkasan_putusan': ringkasan_putusan,            'pasal': pasal,            'pihak': pihak,            'pidana_penjara_tahun': pidana_tahun,            'denda_rp': denda_rp,            'word_count': word_count,            'text_full': text        })        if idx % 10 == 0:            print(f"  --- {idx} selesai ---")    df = pd.DataFrame(cases)    output_path = os.path.join(PROCESSED_DIR, 'cases.csv')    df.to_csv(output_path, index=False, encoding='utf-8')    print(f"\nTersimpan: {output_path}")    print(f"Total kasus: {len(df)}")    print(f"Kolom: {list(df.columns)}")    print("\n=== Ringkasan ===")    print(f"Rata-rata panjang teks: {df['word_count'].mean():.0f} kata")    print(f"Rata-rata pidana: {df['pidana_penjara_tahun'].mean():.1f} tahun")    print(f"Range pidana: {df['pidana_penjara_tahun'].min()} - {df['pidana_penjara_tahun'].max()} tahun")    print(f"Kasus dengan denda: {df[df['denda_rp'] > 0].shape[0]}/{len(df)}")if __name__ == '__main__':    main()Sesi 3: Case Retrieval - TF-IDF Vectorization dan Cosine Similarity (Improved)import osimport reimport jsonimport pickleimport pandas as pdimport numpy as npfrom sklearn.feature_extraction.text import TfidfVectorizerfrom sklearn.metrics.pairwise import cosine_similarityBASE_DIR = os.path.dirname(os.path.abspath(__file__))PROCESSED_DIR = os.path.join(BASE_DIR, '..', 'data', 'processed')EVAL_DIR = os.path.join(BASE_DIR, '..', 'data', 'eval')os.makedirs(EVAL_DIR, exist_ok=True)INDONESIAN_STOPWORDS = [    'dan', 'di', 'ke', 'dari', 'yang', 'ini', 'itu', 'dengan', 'untuk',    'pada', 'adalah', 'akan', 'telah', 'sudah', 'bisa', 'dapat', 'tidak',    'atau', 'juga', 'saya', 'kami', 'kita', 'mereka', 'dia', 'ia',    'oleh', 'sebagai', 'dalam', 'bahwa', 'karena', 'jika', 'maka',    'serta', 'seperti', 'secara', 'tersebut', 'setelah', 'sebelum',    'tentang', 'antara', 'ada', 'lebih', 'lain', 'sangat', 'semua',    'hal', 'sehingga', 'namun', 'tetapi', 'sedangkan', 'meskipun',    'walaupun', 'perlu', 'harus', 'selalu', 'sudah', 'belum', 'pernah',    'saat', 'ketika', 'hingga', 'sampai', 'sejak', 'selama', 'tanpa',    'lagi', 'masih', 'hanya', 'saja', 'pun', 'punya', 'merupakan',    'ialah', 'yakni', 'yaitu', 'baik', 'maupun', 'ataupun',    'membaca', 'menimbang', 'mengingat', 'memperhatikan',    'menetapkan', 'menyatakan', 'menjatuhkan', 'memutuskan',]def preprocess_text(text):    if not isinstance(text, str):        return ''    text = text.lower()    text = re.sub(r'[^\w\s]', ' ', text)    text = re.sub(r'\d+', ' ', text)    text = re.sub(r'\s+', ' ', text)    return text.strip()def retrieve(query_text, vectorizer, tfidf_matrix, case_ids, k=5):    query_clean = preprocess_text(query_text)    query_vec = vectorizer.transform([query_clean])    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()    top_k_idx = similarities.argsort()[-k:][::-1]    results = []    for idx in top_k_idx:        results.append({            'case_id': case_ids[idx],            'score': float(similarities[idx])        })    return resultsdef main():    csv_path = os.path.join(PROCESSED_DIR, 'cases.csv')    df = pd.read_csv(csv_path)    print(f"Loaded {len(df)} cases")    texts = [preprocess_text(str(t)) for t in df['text_full']]    case_ids = df['case_id'].tolist()    print("Membangun TF-IDF vectorizer...")    vectorizer = TfidfVectorizer(        max_features=10000,        min_df=2,        max_df=0.90,        stop_words=INDONESIAN_STOPWORDS,        ngram_range=(1, 3),        sublinear_tf=True    )    tfidf_matrix = vectorizer.fit_transform(texts)    print(f"  Vocabulary size: {len(vectorizer.get_feature_names_out())}")    print(f"  Matrix shape: {tfidf_matrix.shape}")    model_path = os.path.join(EVAL_DIR, 'tfidf_model.pkl')    with open(model_path, 'wb') as f:        pickle.dump({            'vectorizer': vectorizer,            'tfidf_matrix': tfidf_matrix,            'case_ids': case_ids        }, f)    print(f"Model tersimpan: {model_path}")    queries = []    query_sources = [0, 10, 20, 30, 40]  # 5 queries dari case    for i, idx in enumerate(query_sources):        row = df.iloc[idx]        pasal = str(row['pasal']) if pd.notna(row['pasal']) else ''        fakta = str(row['ringkasan_fakta']) if pd.notna(row['ringkasan_fakta']) else ''                ground_truth = [row['case_id']]                nama = str(row['nama_terdakwa']) if pd.notna(row['nama_terdakwa']) else ''        full = str(row['text_full']) if pd.notna(row['text_full']) else ''        query_text = f"{fakta} [Pasal: {pasal}] [Terdakwa: {nama[:50]}]"                query_text = full[:2000]  # 2000 karakter pertama        queries.append({            'query_id': f'query_{i+1:02d}',            'query_text': query_text,            'source_case': row['case_id'],            'ground_truth': ground_truth,            'pasal': pasal,            'type': 'leave_one_out'        })    synthetic_queries = [        {            'query_id': 'query_06',            'query_text': 'Turut serta melakukan melaksanakan penempatan pekerja migran Indonesia. Pekerja migran Indonesia ditempatkan di luar negeri tanpa hak dan dieksploitasi. Pasal 4 juncto Pasal 10 Undang-Undang Nomor 21 Tahun 2007 tentang Pemberantasan Tindak Pidana Perdagangan Orang juncto Pasal 55 Ayat (1) ke-1 KUHP.',            'ground_truth': ['case_004'],            'type': 'synthetic'        },        {            'query_id': 'query_07',            'query_text': 'Terdakwa bekerja sebagai manajer di klub malam yang mempekerjakan anak di bawah umur sebagai waitress. Anak korban dieksploitasi secara ekonomi. Pasal 88 Ayat (1) juncto Pasal 76 I Undang-Undang Perlindungan Anak.',            'ground_truth': ['case_001'],            'type': 'synthetic'        },        {            'query_id': 'query_08',            'query_text': 'Terdakwa menawarkan jasa prostitusi melalui aplikasi MiChat. Terdakwa mencari tamu untuk melakukan persetubuhan dengan para saksi dan mendapatkan fee. Pasal 12 Undang-Undang Nomor 21 Tahun 2007 tentang Pemberantasan Tindak Pidana Perdagangan Orang.',            'ground_truth': ['case_002'],            'type': 'synthetic'        },        {            'query_id': 'query_09',            'query_text': 'Mahasiswa magang dikirim ke Jepang untuk program magang tetapi dipaksa bekerja melebihi jam kerja. Visa diubah dari visa pelajar menjadi visa pekerja. Mereka tidak diberi hak dan gaji yang layak. Pasal 4 juncto Pasal 48 Undang-Undang Nomor 21 Tahun 2007.',            'ground_truth': ['case_070'],            'type': 'synthetic'        },        {            'query_id': 'query_10',            'query_text': 'Terdakwa merekrut anak di bawah umur untuk dieksploitasi secara seksual. Anak korban berusia 16 tahun dan 13 tahun. Terdakwa mencarikan tamu laki-laki untuk anak korban dan mendapatkan fee. Pasal 2 Ayat (1) juncto Pasal 17 Undang-Undang Nomor 21 Tahun 2007.',            'ground_truth': ['case_017'],            'type': 'synthetic'        }    ]    queries.extend(synthetic_queries)    queries_path = os.path.join(EVAL_DIR, 'queries.json')    with open(queries_path, 'w', encoding='utf-8') as f:        json.dump(queries, f, indent=2, ensure_ascii=False)    print(f"Queries tersimpan: {queries_path}")    print("\n=== Uji Retrieval ===")    retrieval_results = []    for q in queries:        results = retrieve(q['query_text'], vectorizer, tfidf_matrix, case_ids, k=5)        retrieved_ids = [r['case_id'] for r in results]        gt = q['ground_truth']        hits = len(set(retrieved_ids) & set(gt))        precision = hits / len(retrieved_ids) if retrieved_ids else 0        recall = hits / len(gt) if gt else 0        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0        mrr = 0        for rank, rid in enumerate(retrieved_ids, 1):            if rid in gt:                mrr = 1.0 / rank                break        retrieval_results.append({            'query_id': q['query_id'],            'type': q['type'],            'precision': precision,            'recall': recall,            'f1': f1,            'mrr': mrr,            'retrieved_ids': retrieved_ids,            'ground_truth': gt        })        print(f"\n{q['query_id']} ({q['type']})")        if q['type'] == 'leave_one_out':            print(f"  Sumber: {q['source_case']}, Ground truth: {gt}")        else:            print(f"  Ground truth: {gt}")        print(f"  Top-5: {retrieved_ids}")        print(f"  P@5: {precision:.2f}, R@5: {recall:.2f}, F1: {f1:.2f}, MRR: {mrr:.3f}")    results_path = os.path.join(EVAL_DIR, 'retrieval_results.json')    with open(results_path, 'w', encoding='utf-8') as f:        json.dump(retrieval_results, f, indent=2, ensure_ascii=False)    for metric in ['precision', 'recall', 'f1', 'mrr']:        vals = [r[metric] for r in retrieval_results]        print(f"\nRata-rata {metric.capitalize()}: {np.mean(vals):.3f}")    metrics_df = pd.DataFrame(retrieval_results)    metrics_path = os.path.join(EVAL_DIR, 'retrieval_metrics.csv')    metrics_df.to_csv(metrics_path, index=False)    print(f"\nMetrik tersimpan: {metrics_path}")if __name__ == '__main__':    main()Sesi 4: Case Solution Reuse - Prediksi Hasil Putusanimport osimport jsonimport pickleimport pandas as pdimport numpy as npBASE_DIR = os.path.dirname(os.path.abspath(__file__))PROCESSED_DIR = os.path.join(BASE_DIR, '..', 'data', 'processed')EVAL_DIR = os.path.join(BASE_DIR, '..', 'data', 'eval')RESULTS_DIR = os.path.join(BASE_DIR, '..', 'data', 'results')os.makedirs(RESULTS_DIR, exist_ok=True)def load_model(eval_dir):    with open(os.path.join(eval_dir, 'tfidf_model.pkl'), 'rb') as f:        model = pickle.load(f)    return model['vectorizer'], model['tfidf_matrix'], model['case_ids']def preprocess_text(text):    import re    if not isinstance(text, str):        return ''    text = text.lower()    text = re.sub(r'[^\w\s]', ' ', text)    text = re.sub(r'\d+', ' ', text)    text = re.sub(r'\s+', ' ', text)    return text.strip()def retrieve(query, vectorizer, tfidf_matrix, case_ids, k=5):    from sklearn.metrics.pairwise import cosine_similarity    query_clean = preprocess_text(query)    query_vec = vectorizer.transform([query_clean])    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()    top_k_idx = similarities.argsort()[-k:][::-1]    results = []    for idx in top_k_idx:        results.append({            'case_id': case_ids[idx],            'score': float(similarities[idx])        })    return resultsdef predict_outcome(query, vectorizer, tfidf_matrix, case_ids, df, k=5):    Prediksi solusi menggunakan Weighted Similarity Voting    results = retrieve(query, vectorizer, tfidf_matrix, case_ids, k)    total_weight = sum(r['score'] for r in results)    if total_weight == 0:        return {            'predicted_pidana': 0,            'predicted_denda': 0,            'top_k_cases': [r['case_id'] for r in results],            'top_k_scores': [r['score'] for r in results],            'method': 'weighted_voting'        }    weighted_pidana = 0.0    weighted_denda = 0.0    for r in results:        case_data = df[df['case_id'] == r['case_id']]        if case_data.empty:            continue        row = case_data.iloc[0]        weight = r['score'] / total_weight        weighted_pidana += weight * float(row['pidana_penjara_tahun'])        weighted_denda += weight * float(row['denda_rp'])    return {        'predicted_pidana': round(weighted_pidana, 1),        'predicted_denda': round(weighted_denda, -3),  # Dibulatkan ke ribuan        'top_k_cases': [r['case_id'] for r in results],        'top_k_scores': [r['score'] for r in results],        'method': 'weighted_voting'    }def main():    csv_path = os.path.join(PROCESSED_DIR, 'cases.csv')    df = pd.read_csv(csv_path)    queries_path = os.path.join(EVAL_DIR, 'queries.json')    with open(queries_path, 'r', encoding='utf-8') as f:        queries = json.load(f)    vectorizer, tfidf_matrix, case_ids = load_model(EVAL_DIR)    print(f"Model loaded. {len(case_ids)} cases in corpus.")    print(f"\n=== Prediksi untuk {len(queries)} Queries ===\n")    predictions = []    for q in queries:        qid = q['query_id']        result = predict_outcome(q['query_text'], vectorizer, tfidf_matrix, case_ids, df, k=5)        ground_pidana = 0        ground_denda = 0        if q['type'] == 'leave_one_out':            source = q['source_case']            src = df[df['case_id'] == source]            if not src.empty:                ground_pidana = int(src.iloc[0]['pidana_penjara_tahun'])                ground_denda = int(src.iloc[0]['denda_rp'])        else:            gt = q['ground_truth']            if gt:                gt_data = df[df['case_id'].isin(gt)]                if not gt_data.empty:                    ground_pidana = int(gt_data.iloc[0]['pidana_penjara_tahun'])                    ground_denda = int(gt_data.iloc[0]['denda_rp'])        predictions.append({            'query_id': qid,            'type': q['type'],            'predicted_pidana': result['predicted_pidana'],            'predicted_denda': result['predicted_denda'],            'ground_truth_pidana': ground_pidana,            'ground_truth_denda': ground_denda,            'top_5_case_ids': '; '.join(result['top_k_cases']),            'top_5_scores': '; '.join([f"{s:.4f}" for s in result['top_k_scores']]),            'method': result['method']        })        print(f"{qid} ({q['type']})")        print(f"  Prediksi: {result['predicted_pidana']} thn, Rp{result['predicted_denda']:,.0f}")        print(f"  Ground:   {ground_pidana} thn, Rp{ground_denda:,.0f}")        print(f"  Top-5: {result['top_k_cases']}")        print(f"  Skor: {[round(s, 4) for s in result['top_k_scores']]}")        print()    pred_df = pd.DataFrame(predictions)    output_path = os.path.join(RESULTS_DIR, 'predictions.csv')    pred_df.to_csv(output_path, index=False, encoding='utf-8')    print(f"Prediksi tersimpan: {output_path}")    valid = pred_df[pred_df['ground_truth_pidana'] > 0]    if not valid.empty:        mae_pidana = abs(valid['predicted_pidana'] - valid['ground_truth_pidana']).mean()        print(f"\nMAE Pidana: {mae_pidana:.2f} tahun")        mae_denda = abs(valid['predicted_denda'] - valid['ground_truth_denda']).mean()        print(f"MAE Denda: Rp{mae_denda:,.0f}")    return pred_dfif __name__ == '__main__':    main()Sesi 5: Evaluasi Model - Metrik Retrieval dan Prediksiimport osimport jsonimport pandas as pdimport numpy as npfrom sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_errorBASE_DIR = os.path.dirname(os.path.abspath(__file__))EVAL_DIR = os.path.join(BASE_DIR, '..', 'data', 'eval')RESULTS_DIR = os.path.join(BASE_DIR, '..', 'data', 'results')PROCESSED_DIR = os.path.join(BASE_DIR, '..', 'data', 'processed')def evaluate_retrieval():    retrieval_path = os.path.join(EVAL_DIR, 'retrieval_results.json')    with open(retrieval_path, 'r', encoding='utf-8') as f:        results = json.load(f)    metrics = []    for r in results:        retrieved = set(r['retrieved_ids'])        ground = set(r['ground_truth'])        true_pos = len(retrieved & ground)        false_pos = len(retrieved - ground)        false_neg = len(ground - retrieved)        precision = true_pos / (true_pos + false_pos) if (true_pos + false_pos) > 0 else 0        recall = true_pos / (true_pos + false_neg) if (true_pos + false_neg) > 0 else 0        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0        mrr = 0        for rank, cid in enumerate(r['retrieved_ids'], 1):            if cid in ground:                mrr = 1.0 / rank                break        metrics.append({            'query_id': r['query_id'],            'type': r['type'],            'precision@5': round(precision, 3),            'recall@5': round(recall, 3),            'f1_score': round(f1, 3),            'mrr': round(mrr, 3),            'top1_match': 'YES' if mrr > 0 else 'NO'        })    df = pd.DataFrame(metrics)    output_path = os.path.join(EVAL_DIR, 'retrieval_metrics.csv')    df.to_csv(output_path, index=False)    print("=== METRIK RETRIEVAL ===")    print(df.to_string(index=False))    print(f"\nRata-rata:")    for col in ['precision@5', 'recall@5', 'f1_score', 'mrr']:        print(f"  {col}: {df[col].mean():.3f}")    print(f"  Query dengan top-1 match: {df['top1_match'].value_counts().get('YES', 0)}/{len(df)}")    return dfdef evaluate_prediction():    pred_path = os.path.join(RESULTS_DIR, 'predictions.csv')    df = pd.read_csv(pred_path)    print("\n=== METRIK PREDIKSI ===")    valid = df[df['ground_truth_pidana'] > 0].copy()    if valid.empty:        print("Tidak ada data valid untuk evaluasi prediksi")        return df    mae_pidana = mean_absolute_error(valid['ground_truth_pidana'], valid['predicted_pidana'])    mae_denda = mean_absolute_error(valid['ground_truth_denda'], valid['predicted_denda'])    print(f"MAE Pidana Penjara: {mae_pidana:.2f} tahun")    print(f"MAE Denda: Rp{mae_denda:,.0f}")    valid['error_pidana'] = abs(valid['predicted_pidana'] - valid['ground_truth_pidana'])    valid['error_denda'] = abs(valid['predicted_denda'] - valid['ground_truth_denda'])    print(f"\nError Analysis:")    print(f"  Error pidana <= 2 tahun: {(valid['error_pidana'] <= 2).sum()}/{len(valid)}")    print(f"  Error pidana <= 5 tahun: {(valid['error_pidana'] <= 5).sum()}/{len(valid)}")    metrics = valid[['query_id', 'type', 'predicted_pidana', 'ground_truth_pidana',                     'predicted_denda', 'ground_truth_denda', 'error_pidana', 'error_denda']].copy()    metrics_path = os.path.join(EVAL_DIR, 'prediction_metrics.csv')    metrics.to_csv(metrics_path, index=False)    print(f"\nMetrik prediksi tersimpan: {metrics_path}")    print("\n=== ANALISIS KEGAGALAN ===")    worst = valid.nlargest(3, 'error_pidana')    for _, row in worst.iterrows():        print(f"\n{row['query_id']} ({row['type']})")        print(f"  Prediksi: {row['predicted_pidana']} thn, Ground: {row['ground_truth_pidana']} thn")        print(f"  Error: {row['error_pidana']:.1f} tahun")        print(f"  Top-5: {row['top_5_case_ids']}")    print("\nPenyebab potensial kegagalan:")    print("1. Ringkasan fakta query tidak memiliki cukup kesamaan dengan kasus target")    print("2. Kasus target memiliki pasal yang unik/berbeda dari retrieved cases")    print("3. Terdapat noise pada ekstraksi pidana dari teks (parsing error)")    print("4. Beberapa kasus memiliki pidana 0 (bebas) yang mempengaruhi rata-rata")    return dfdef main():    print("EVALUASI SISTEM CBR - ANALISIS PUTUSAN PENGADILAN")    print("=" * 50)    ret_metrics = evaluate_retrieval()    print("\n")    pred_metrics = evaluate_prediction()    print("\n" + "=" * 50)    print("RINGKASAN EVALUASI")    print("=" * 50)    print(f"Total kasus dalam case base: 70")    print(f"Total query uji: 10")    print(f"  - Leave-one-out: 5")    print(f"  - Skenario sintetis: 5")    avg_prec = ret_metrics['precision@5'].mean()    avg_rec = ret_metrics['recall@5'].mean()    avg_f1 = ret_metrics['f1_score'].mean()    avg_mrr = ret_metrics['mrr'].mean()    print(f"\nRetrieval:")    print(f"  Precision@5: {avg_prec:.3f}")    print(f"  Recall@5:    {avg_rec:.3f}")    print(f"  F1-Score:    {avg_f1:.3f}")    print(f"  MRR:         {avg_mrr:.3f}")    pred_csv = os.path.join(RESULTS_DIR, 'predictions.csv')    pred_df = pd.read_csv(pred_csv)    valid_pred = pred_df[pred_df['ground_truth_pidana'] > 0]    if not valid_pred.empty:        mae_p = mean_absolute_error(valid_pred['ground_truth_pidana'], valid_pred['predicted_pidana'])        mae_d = mean_absolute_error(valid_pred['ground_truth_denda'], valid_pred['predicted_denda'])        print(f"\nPrediksi:")        print(f"  MAE Pidana: {mae_p:.2f} tahun")        print(f"  MAE Denda: Rp{mae_d:,.0f}")if __name__ == '__main__':    main()